# SE3_model_train_validate


<span style='color:red'>delete before publishing</span>

# About this notebook (for developer, delete this cell)

This notebook is the third step in the DIWA reproducible research pipeline. It reads **analysis-ready data** produced by SE2 and trains, calibrates, and validates a model in the water resources domain.

Supported model types include:
- **Statistical / hypothesis-testing models** — trend tests (Mann-Kendall, Spearman), change-point detection, frequency analysis, regression
- **Data-driven / machine learning models** — Random Forest, XGBoost, LSTM, Gaussian Process Regression
- **Process-based / conceptual models** — HBV, HydroSHEDS, lumped rainfall-runoff, SWAT wrappers

**Key design rules (do not violate):**
- No data is held in memory between SE notebooks. All inputs are read from disk; all outputs are written to disk.
- Trained model artefacts (weights, parameters, serialised objects) are saved in `model/` inside the processed data directory.
- Model configuration (hyperparameters, feature lists, split dates) is stored as a YAML or JSON file in `model/config/`.
- Predictions and evaluation outputs are saved to `autobag_processeddata/` outside the repository.
- Validation follows the framework of **Gagne et al. (2023)** *Artificial Intelligence for the Earth Systems* (AIES-D-22-0065): use held-out test sets, report uncertainty, avoid data leakage, and compare against a meaningful baseline.

**Recommended packages by model type:**

| Model type | Packages |
|---|---|
| Statistical tests | `scipy`, `pymannkendall`, `ruptures`, `statsmodels` |
| Machine learning | `scikit-learn`, `xgboost`, `lightgbm`, `torch`, `tensorflow` |
| Time-series deep learning | `pytorch-forecasting`, `neuralforecast`, `darts` |
| Hydrological process models | `HydPy`, `pyet`, `hydroeval`, `spotpy` |
| Model evaluation | `hydroeval`, `hydrotools`, `HydroErr` |
| Calibration / uncertainty | `spotpy`, `lmfit`, `emcee`, `SALib` |
| Config management | `pyyaml`, `hydra-core` |
| Model serialisation | `joblib`, `pickle`, `onnx` |

In [ ]:
#Modify cell
# Load dependencies — keep only what you use, add what you need
from pathlib import Path
import pandas as pd
import numpy as np
import yaml
import json
import joblib
import logging

# Statistical testing
from scipy import stats
import pymannkendall as mk          # pip install pymannkendall

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

# Hydrological evaluation metrics
import hydroeval as he               # pip install hydroeval

# Plotting
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

<span style='color:goldenrod'>edit before publishing</span>

# Overview

## Model description
Describe the type of model used and its scientific motivation:
- What process or pattern does the model represent?
- Why is this model appropriate for the water resources question?
- What are the key assumptions?

## Input data
Describe analysis-ready data produced by SE2:
- File names and formats
- Predictor variables (features) and target variable(s)
- Temporal and spatial resolution
- Any known data quality issues

## Outputs
Describe what this notebook produces:
- Trained model artefact saved to `model/` subfolder
- Model configuration file saved to `model/config/`
- Predictions saved to `autobag_processeddata/predictions/`
- Evaluation metrics and plots saved to `autobag_processeddata/predictions/`

## Calibration / training strategy
Describe your train / validation / test split strategy:
- Date ranges or fractions used for each split
- Rationale for split (temporal, spatial, or random)
- Any cross-validation scheme applied

## Validation framework (Gagne et al. 2023)
Describe how validation follows best practices for AI/ML in Earth systems:
- Baseline model used for comparison (e.g. climatological mean, persistence, simple regression)
- Metrics reported and why they were chosen
- How data leakage was prevented
- Uncertainty quantification approach

<span style='color:green'>keep</span>

## Directory setup
All data are read from and written to directories **outside the repository**. Model artefacts are stored in `model/` and predictions in `predictions/` within the processed data directory. No data objects are passed between notebooks in memory.

In [ ]:
# Automatically generated cell — resolve external directories
# Current working directory = project/notebooks
notebook_dir = Path.cwd()

# Project root (one level up from notebooks)
project_root = notebook_dir.parent

# Parent directory OUTSIDE the repository
external_base = project_root.parent

# Read processed data produced by SE2
proc_data_dir = external_base / "autobag_processeddata"

# Model artefacts: config + serialised model
model_dir     = proc_data_dir / "model"
config_dir    = model_dir / "config"

# Predictions and evaluation outputs
pred_dir      = proc_data_dir / "predictions"

for d in [model_dir, config_dir, pred_dir]:
    d.mkdir(parents=True, exist_ok=True)

print("Processed data directory : ", proc_data_dir.resolve())
print("Model directory          : ", model_dir.resolve())
print("Config directory         : ", config_dir.resolve())
print("Predictions directory    : ", pred_dir.resolve())

In [ ]:
!df -k ..
# Confirm sufficient storage is available before writing model outputs

<span style='color:red'>delete before publishing</span>

# Step 1 — Define and save model configuration

Saving all hyperparameters, feature lists, and split dates to a versioned config file before training ensures reproducibility. Anyone re-running SE3 can inspect exactly what settings produced the stored model artefact. Use YAML for human readability.

**Modify the config below to match your model.** Delete the config entries that do not apply to your model type.

In [ ]:
#Modify cell
config = {
    "model_type": "RandomForestRegressor",   # e.g. RandomForest, LSTM, HBV, MannKendall
    "target_variable": "streamflow_m3s",      # column name of the variable being predicted / tested
    "features": [                             # list of predictor column names from SE2 output
        "precipitation_mm",
        "temperature_C",
        "snowmelt_mm",
    ],
    "train_period": {"start": "1990-01-01", "end": "2005-12-31"},
    "validation_period": {"start": "2006-01-01", "end": "2010-12-31"},
    "test_period": {"start": "2011-01-01", "end": "2020-12-31"},
    "hyperparameters": {
        "n_estimators": 200,
        "max_depth": 10,
        "random_state": 42,
    },
    "baseline_model": "climatological_mean",  # model used as benchmark for skill score
    "scale_features": True,
    "input_file": "analysis_ready_data.csv",  # filename inside proc_data_dir
    "model_version": "v1.0",
}

config_path = config_dir / "model_config.yaml"
with open(config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Config saved to {config_path}")

<span style='color:red'>delete before publishing</span>

# Step 2 — Load analysis-ready data from disk

Always read data from the file written by SE2. Never pass objects directly between notebooks. This enforces the no-RAM-sharing rule and ensures SE3 can be re-run independently.

In [ ]:
#Modify cell
# Load config (re-load from file so SE3 is self-contained)
with open(config_path) as f:
    cfg = yaml.safe_load(f)

# Read analysis-ready data written by SE2
input_path = proc_data_dir / cfg["input_file"]
df = pd.read_csv(input_path, parse_dates=["date"], index_col="date")

print(f"Loaded: {input_path}")
print(f"Shape : {df.shape}")
print(f"Period: {df.index.min()} to {df.index.max()}")
df.head()

<span style='color:red'>delete before publishing</span>

# Step 3 — Train / validation / test split

Use temporal splitting for time-series data to prevent data leakage (future information must never influence training). For spatial generalisation studies, split by catchment or region instead of time.

In [ ]:
#Modify cell
features = cfg["features"]
target   = cfg["target_variable"]

train_start = cfg["train_period"]["start"]
train_end   = cfg["train_period"]["end"]
val_start   = cfg["validation_period"]["start"]
val_end     = cfg["validation_period"]["end"]
test_start  = cfg["test_period"]["start"]
test_end    = cfg["test_period"]["end"]

train = df.loc[train_start:train_end]
val   = df.loc[val_start:val_end]
test  = df.loc[test_start:test_end]

X_train, y_train = train[features], train[target]
X_val,   y_val   = val[features],   val[target]
X_test,  y_test  = test[features],  test[target]

print(f"Train: {len(train)} samples ({train_start} – {train_end})")
print(f"Val  : {len(val)}   samples ({val_start} – {val_end})")
print(f"Test : {len(test)}  samples ({test_start} – {test_end})")

# Check for NaN before training
for name, X in [("Train", X_train), ("Val", X_val), ("Test", X_test)]:
    n_nan = X.isna().sum().sum()
    if n_nan > 0:
        logging.warning(f"{name} features contain {n_nan} NaN values — address in SE2")

<span style='color:red'>delete before publishing</span>

# Step 4 — Feature scaling (if required)

Scale features using statistics computed **only on training data**. Fitting the scaler on the full dataset leaks information from validation/test periods into training — a common source of overly optimistic performance.

In [ ]:
#Modify cell
if cfg["scale_features"]:
    scaler = StandardScaler()
    X_train_sc = scaler.fit_transform(X_train)   # fit ONLY on training data
    X_val_sc   = scaler.transform(X_val)
    X_test_sc  = scaler.transform(X_test)

    # Save scaler alongside model
    joblib.dump(scaler, model_dir / "feature_scaler.joblib")
    print("Scaler saved to", model_dir / "feature_scaler.joblib")
else:
    X_train_sc, X_val_sc, X_test_sc = X_train.values, X_val.values, X_test.values

<span style='color:red'>delete before publishing</span>

# Step 5 — Train model

Replace the example below with your model. Keep this section focused on fitting; evaluation is in Step 7.

**For process-based models**: replace with calibration code (e.g. using `spotpy` or `lmfit`). Save calibrated parameters as the model artefact.

**For statistical hypothesis tests**: replace with the test call (e.g. `mk.original_test()`). Save the result object.

In [ ]:
#Modify cell
hp = cfg["hyperparameters"]

model = RandomForestRegressor(
    n_estimators=hp["n_estimators"],
    max_depth=hp["max_depth"],
    random_state=hp["random_state"],
    n_jobs=-1,
)
model.fit(X_train_sc, y_train)

# Save trained model artefact
model_path = model_dir / f"trained_model_{cfg['model_version']}.joblib"
joblib.dump(model, model_path)
print(f"Trained model saved to {model_path}")

<span style='color:red'>delete before publishing</span>

# Step 6 — Generate and save predictions

Predictions for all three splits are saved to the `predictions/` directory. SE4 reads from this directory to generate figures; SE3 writes nothing to the repository.

In [ ]:
#Modify cell
y_pred_train = model.predict(X_train_sc)
y_pred_val   = model.predict(X_val_sc)
y_pred_test  = model.predict(X_test_sc)

# Assemble dataframes for each split
def _pred_df(index, observed, predicted, split_name):
    return pd.DataFrame({
        "observed":  observed.values,
        "predicted": predicted,
        "split":     split_name,
    }, index=index)

df_pred = pd.concat([
    _pred_df(train.index, y_train, y_pred_train, "train"),
    _pred_df(val.index,   y_val,   y_pred_val,   "validation"),
    _pred_df(test.index,  y_test,  y_pred_test,  "test"),
])
df_pred.index.name = "date"

pred_path = pred_dir / "predictions.csv"
df_pred.to_csv(pred_path)
print(f"Predictions saved to {pred_path}")

<span style='color:red'>delete before publishing</span>

# Step 7 — Model validation

Validation follows the framework recommended by **Gagne et al. (2023)** for AI/ML models in Earth systems:

1. **Report multiple metrics** spanning bias, magnitude error, and correlation (NSE, KGE, RMSE, MAE, R²).
2. **Compare against a baseline** — always evaluate skill relative to a naïve reference model.
3. **Report separately for each split** — training performance alone is uninformative.
4. **Evaluate across the full flow range** — log-NSE downweights high flows; consider split metrics for low/high flow regimes.
5. **Save all metric results to disk** for use in SE4 figures and README tables.

In [ ]:
#Modify cell
def compute_metrics(observed, predicted, split_name):
    """Compute a standard suite of hydrological model performance metrics."""
    obs  = np.array(observed)
    sim  = np.array(predicted)

    nse  = he.evaluator(he.nse,  sim, obs)[0]
    kge  = he.evaluator(he.kge,  sim, obs)[0][0]   # KGE (Kling-Gupta Efficiency)
    pbias = he.evaluator(he.pbias, sim, obs)[0]
    rmse = np.sqrt(mean_squared_error(obs, sim))
    r2   = r2_score(obs, sim)

    # Log-NSE (emphasises low flows)
    log_obs = np.log(obs + 1e-6)
    log_sim = np.log(sim + 1e-6)
    log_nse = 1 - np.sum((log_obs - log_sim)**2) / np.sum((log_obs - log_obs.mean())**2)

    return {
        "split":   split_name,
        "NSE":     round(nse, 4),
        "KGE":     round(kge, 4),
        "PBIAS_%": round(pbias, 2),
        "RMSE":    round(rmse, 4),
        "R2":      round(r2, 4),
        "logNSE":  round(log_nse, 4),
    }

metrics_rows = [
    compute_metrics(y_train, y_pred_train, "train"),
    compute_metrics(y_val,   y_pred_val,   "validation"),
    compute_metrics(y_test,  y_pred_test,  "test"),
]

# Baseline: climatological mean (training-period mean applied to all splits)
baseline_pred = np.full_like(y_test.values, fill_value=y_train.mean(), dtype=float)
metrics_rows.append(compute_metrics(y_test, baseline_pred, "baseline_clim_mean"))

metrics_df = pd.DataFrame(metrics_rows).set_index("split")
print(metrics_df.to_string())

# Save metrics to disk
metrics_path = pred_dir / "evaluation_metrics.csv"
metrics_df.to_csv(metrics_path)
print(f"\nMetrics saved to {metrics_path}")

<span style='color:red'>delete before publishing</span>

# Step 8 — Diagnostic validation plots

Visualisation follows the ten guidelines of **Kelleher & Wagener (2011)**:
- Use the simplest graph that conveys the information (Guideline 1).
- Use position and length to encode quantitative values (Guideline 2).
- Focus on patterns vs. details as appropriate (Guideline 3).
- Label axes clearly with units; avoid chart junk (Guidelines 5 & 7).
- Use colour to highlight, not to decorate; be colourblind-friendly (Guideline 8).

Plots are saved to `predictions/` for use in SE4.

In [ ]:
#Modify cell
# ── Plot 1: Hydrograph — observed vs. predicted (test period) ─────────────────
# Guideline 1: simple line plot; no 3D, no dual-axis decorations
# Guideline 3: emphasise temporal patterns, not individual points
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test.index, y_test.values,  color="#2166ac", lw=1.2, label="Observed", zorder=3)
ax.plot(test.index, y_pred_test,    color="#d6604d", lw=1.0, label="Predicted", zorder=2)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.set_xlabel("Date")
ax.set_ylabel(f"{target}")
ax.set_title(f"Hydrograph — test period  |  NSE={metrics_df.loc['test','NSE']:.3f}  KGE={metrics_df.loc['test','KGE']:.3f}")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(pred_dir / "hydrograph_test.png", dpi=150)
plt.show()
print("Hydrograph saved")

In [ ]:
#Modify cell
# ── Plot 2: Scatter — observed vs. predicted (all splits) ─────────────────────
# Guideline 2: position encodes value (scatter) — best for quantitative accuracy
# Guideline 8: distinct colourblind-safe palette per split
split_colours = {"train": "#4dac26", "validation": "#f1a340", "test": "#d01c8b"}
fig, ax = plt.subplots(figsize=(5, 5))
for sp, col in split_colours.items():
    mask = df_pred["split"] == sp
    ax.scatter(df_pred.loc[mask, "observed"], df_pred.loc[mask, "predicted"],
               s=6, alpha=0.4, color=col, label=sp, rasterized=True)
lims = [df_pred[["observed", "predicted"]].min().min(),
        df_pred[["observed", "predicted"]].max().max()]
ax.plot(lims, lims, "k--", lw=0.8, label="1:1 line")
ax.set_xlabel(f"Observed {target}")
ax.set_ylabel(f"Predicted {target}")
ax.set_title("Observed vs. Predicted")
ax.legend(frameon=False, markerscale=2)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(pred_dir / "scatter_obs_pred.png", dpi=150)
plt.show()
print("Scatter plot saved")

In [ ]:
#Modify cell
# ── Plot 3: Flow Duration Curve — observed vs. predicted (test period) ─────────
# Guideline 3: FDC is a pattern plot — reveals performance across flow regime
# Guideline 5: log-y scale appropriate for streamflow spanning orders of magnitude
def fdc(series):
    sorted_vals = np.sort(series)[::-1]
    exceedance  = np.arange(1, len(sorted_vals) + 1) / (len(sorted_vals) + 1)
    return exceedance, sorted_vals

fig, ax = plt.subplots(figsize=(6, 4))
exc_obs,  q_obs  = fdc(y_test.values)
exc_pred, q_pred = fdc(y_pred_test)
ax.semilogy(exc_obs  * 100, q_obs,  color="#2166ac", lw=1.2, label="Observed")
ax.semilogy(exc_pred * 100, q_pred, color="#d6604d", lw=1.0, label="Predicted", ls="--")
ax.set_xlabel("Exceedance probability (%)")
ax.set_ylabel(f"{target} (log scale)")
ax.set_title("Flow Duration Curve — test period")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(pred_dir / "fdc_test.png", dpi=150)
plt.show()
print("FDC saved")

<span style='color:red'>delete before publishing</span>

# Step 9 — Feature importance (data-driven models only)

Feature importance analysis supports interpretability. For process-based models, replace this cell with a parameter sensitivity analysis (e.g. using `SALib`). Delete this section for hypothesis-testing models.

In [ ]:
#Modify cell — delete if not using a feature-importance-capable model
importances = pd.Series(model.feature_importances_, index=features).sort_values()

# Guideline 1 & 2: horizontal bar chart — position encodes importance value
fig, ax = plt.subplots(figsize=(5, len(features) * 0.5 + 1))
importances.plot.barh(ax=ax, color="#4393c3", edgecolor="none")
ax.set_xlabel("Feature importance (mean decrease impurity)")
ax.set_title("Feature importance")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(pred_dir / "feature_importance.png", dpi=150)
plt.show()
print("Feature importance plot saved")

<span style='color:goldenrod'>edit before publishing</span>

# Summary of results

Summarise model performance here in plain language:
- How does the model perform on the test set relative to the baseline?
- Are there systematic biases (e.g. overestimation of low flows)?
- Does the FDC reveal performance differences across the flow regime?
- What are the main sources of uncertainty or error?

## Files written by this notebook

| File | Location | Description |
|---|---|---|
| `model_config.yaml` | `model/config/` | All hyperparameters, features, and split dates |
| `trained_model_v1.0.joblib` | `model/` | Serialised trained model |
| `feature_scaler.joblib` | `model/` | Fitted feature scaler |
| `predictions.csv` | `predictions/` | Observed and predicted values for all splits |
| `evaluation_metrics.csv` | `predictions/` | Performance metrics for all splits + baseline |
| `hydrograph_test.png` | `predictions/` | Time-series diagnostic plot |
| `scatter_obs_pred.png` | `predictions/` | Observed vs. predicted scatter |
| `fdc_test.png` | `predictions/` | Flow duration curve comparison |
| `feature_importance.png` | `predictions/` | Feature importance bar chart |

## Need help?

Search, ask, and answer modelling questions at https://github.com/orgs/DigitalWaters-fi/discussions

Tag #modelling